# DE2 — Assignment 2: Text — Inverted Index
Author: JABRY — ESIEE 2025-2026 — Data Engineering II
Track: A — Esports (hero lore + patch notes + match commentary)
Goal: Build a text-processing pipeline producing an inverted index in Parquet, measure query latency and disk footprint, and prove choices with plans and metrics.

## 0. Setup

In [1]:
import os
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
import time, pathlib, csv, io, contextlib, shutil
from datetime import datetime

DE2_SPARK_DRIVER_HOST = os.environ.get("DE2_SPARK_DRIVER_HOST", "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)


def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows browser):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")

spark = SparkSession.builder \
    .appName("de2-assignment2") \
    .master("local[*]") \
    .config("spark.driver.host", DE2_SPARK_DRIVER_HOST) \
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.ui.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .getOrCreate()

show_spark_ui(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/11 13:49:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows browser): http://localhost:4040


## 1. Corpus Ingestion (Track A — Esports)

In [2]:
schema = StructType([
    StructField("doc_id", StringType(), False),
    StructField("text",   StringType(), True),
])

df_corpus = spark.read.schema(schema).option("header", "true").csv("data/corpus/esports_corpus.csv")
n_docs = df_corpus.count()
print(f"Documents: {n_docs}")
df_corpus = df_corpus.withColumn("doc_len", F.length("text"))
avg_len = df_corpus.select(F.avg('doc_len')).first()[0]
print(f"Avg doc length: {avg_len:.0f} chars")
df_corpus.show(5, truncate=80)

Documents: 150
Avg doc length: 228 chars
+--------+--------------------------------------------------------------------------------+-------+
|  doc_id|                                                                            text|doc_len|
+--------+--------------------------------------------------------------------------------+-------+
|hero_001|The Dragon Knight is a tanky melee hero who uses fire breath as his ultimate ...|    214|
|hero_002|Crystal Maiden is a ranged support hero with strong mana regeneration. Her ul...|    200|
|hero_003|Invoker is a complex intelligence hero with ten different abilities combined ...|    215|
|hero_004|Phantom Assassin is an agility carry with critical strike passive ability. He...|    194|
|hero_005|Pudge is a strength hero famous for his hook ability that pulls enemies. His ...|    198|
+--------+--------------------------------------------------------------------------------+-------+
only showing top 5 rows


## 2. Text Normalization — Lowercase, Tokenize, Remove Stop-Words

In [3]:
df_clean = df_corpus.withColumn("text_clean",
    F.regexp_replace(F.lower(F.col("text")), r"[^a-z0-9\s]", ""))

df_tokens = df_clean.withColumn("tokens", F.split("text_clean", r"\s+"))
df_exploded = df_tokens.select("doc_id", F.explode("tokens").alias("token"))
df_exploded = df_exploded.filter(F.length("token") > 0)
tokens_before = df_exploded.count()
print(f"Total tokens before stop-word removal: {tokens_before}")

STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "in", "on", "at",
              "to", "for", "of", "and", "or", "not", "it", "this", "that", "with",
              "by", "as", "from", "be", "has", "have", "had", "but", "if", "then"}
df_filtered = df_exploded.filter(~F.col("token").isin(STOP_WORDS))
tokens_after = df_filtered.count()
print(f"Tokens after stop-word removal:  {tokens_after}")
print(f"Removed {tokens_before - tokens_after} stop-word occurrences")

Total tokens before stop-word removal: 5388
Tokens after stop-word removal:  3864
Removed 1524 stop-word occurrences


## 3. Build Inverted Index

In [4]:
inverted_index = (df_filtered
    .groupBy("token")
    .agg(
        F.collect_list("doc_id").alias("doc_ids"),
        F.count("*").alias("freq")
    )
    .orderBy(F.desc("freq")))

n_unique = inverted_index.count()
print(f"Unique terms: {n_unique}")
inverted_index.show(20, truncate=60)

# Save the index build plan as evidence
pathlib.Path("proof").mkdir(parents=True, exist_ok=True)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    inverted_index.explain("formatted")
with open("proof/plan_index_build.txt", "w") as f:
    f.write(buf.getvalue())
print("Plan saved to proof/plan_index_build.txt")

Unique terms: 265
+----------+------------------------------------------------------------+----+
|     token|                                                     doc_ids|freq|
+----------+------------------------------------------------------------+----+
|      hero|[hero_009, hero_010, hero_010, patch_011, patch_012, patc...| 107|
|      team|[match_101, match_101, match_102, match_102, match_103, m...| 103|
|   ability|[hero_010, patch_011, patch_012, patch_013, patch_014, pa...|  69|
| different|[patch_011, patch_012, patch_013, patch_014, patch_015, p...|  61|
|    expect|[patch_011, patch_012, patch_013, patch_014, patch_015, p...|  60|
|      aims|[patch_011, patch_012, patch_013, patch_014, patch_015, p...|  60|
|      been|[patch_011, patch_012, patch_013, patch_014, patch_015, p...|  60|
|   changed|[patch_011, patch_012, patch_013, patch_014, patch_015, p...|  60|
|   current|[patch_011, patch_012, patch_013, patch_014, patch_015, p...|  60|
|     bring|[patch_011, patch_012,

## 4. Write Inverted Index — Parquet and CSV

In [5]:
for path in ["outputs/lab2/inverted_index", "outputs/lab2/inverted_index_csv"]:
    if pathlib.Path(path).exists():
        shutil.rmtree(path)

pathlib.Path("outputs/lab2/inverted_index").mkdir(parents=True, exist_ok=True)
pathlib.Path("outputs/lab2/inverted_index_csv").mkdir(parents=True, exist_ok=True)

inverted_index.write.mode("overwrite").parquet("outputs/lab2/inverted_index")

inverted_index.withColumn("doc_ids", F.concat_ws(",", "doc_ids")) \
    .write.mode("overwrite").option("header", "true").csv("outputs/lab2/inverted_index_csv")

print("Parquet and CSV written.")

Parquet and CSV written.


## 5. Query Latency Measurement

In [6]:
idx = spark.read.parquet("outputs/lab2/inverted_index")
idx.cache()
idx.count()  # force cache

query_terms = ["hero", "ability", "damage", "mana", "ultimate"]
query_results = []

for term in query_terms:
    t0 = time.time()
    result = idx.filter(F.col("token") == term).collect()
    latency_ms = (time.time() - t0) * 1000
    if result:
        freq = result[0]['freq']
        n_post = len(result[0]['doc_ids'])
        print(f"'{term}': df={freq}, postings={n_post}, latency={latency_ms:.1f} ms")
        query_results.append((term, freq, n_post, latency_ms))
    else:
        print(f"'{term}': NOT FOUND, latency={latency_ms:.1f} ms")
        query_results.append((term, 0, 0, latency_ms))

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    idx.filter(F.col("token") == "hero").explain("formatted")
with open("proof/plan_query.txt", "w") as f:
    f.write(buf.getvalue())
print("\nQuery plan saved to proof/plan_query.txt")

'hero': df=107, postings=107, latency=108.2 ms
'ability': df=69, postings=69, latency=54.2 ms
'damage': df=53, postings=53, latency=47.5 ms
'mana': df=15, postings=15, latency=39.8 ms
'ultimate': df=59, postings=59, latency=49.3 ms

Query plan saved to proof/plan_query.txt


## 6. Storage Footprint Comparison — Parquet vs CSV

In [7]:
def dir_size(path):
    return sum(f.stat().st_size for f in pathlib.Path(path).rglob("*") if f.is_file())

def count_files(path):
    return sum(1 for f in pathlib.Path(path).rglob("*") if f.is_file() and not f.name.startswith(".") and not f.name.startswith("_"))

parquet_bytes = dir_size("outputs/lab2/inverted_index")
csv_bytes = dir_size("outputs/lab2/inverted_index_csv")
ratio = parquet_bytes / csv_bytes if csv_bytes > 0 else 0

parquet_files = count_files("outputs/lab2/inverted_index")
csv_files = count_files("outputs/lab2/inverted_index_csv")

print(f"Parquet: {parquet_bytes:,} bytes  ({parquet_files} files)")
print(f"CSV:     {csv_bytes:,} bytes  ({csv_files} files)")
print(f"Ratio (Parquet/CSV): {ratio:.2%}")

Parquet: 5,676 bytes  (1 files)
CSV:     40,767 bytes  (1 files)
Ratio (Parquet/CSV): 13.92%


## 7. Evidence & Metrics — `lab2_metrics_log.csv`

In [8]:
header = ["run_id", "task", "input_size_bytes", "output_size_bytes", "files_read",
          "latency_ms", "freq", "postings", "shuffle_read_bytes_ui", "shuffle_write_bytes_ui",
          "timestamp", "notes"]

rows = []
now = datetime.now().isoformat()

rows.append([1, "index_build", "", parquet_bytes, parquet_files, "", n_unique, "", "", "", now, f"unique tokens={n_unique} from {tokens_after} filtered tokens"])
rows.append([2, "footprint_parquet", "", parquet_bytes, parquet_files, "", "", "", "", "", now, f"{parquet_files} parquet files"])
rows.append([3, "footprint_csv",     "", csv_bytes,     csv_files,     "", "", "", "", "", now, f"{csv_files} csv files, ratio parquet/csv={ratio:.2%}"])

for i, (term, freq, n_post, lat) in enumerate(query_results, start=4):
    rows.append([i, f"query_lookup_{term}", "", "", "", round(lat, 2), freq, n_post, "", "", now, "latency wall-clock from time.time()"])

with open("lab2_metrics_log.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(header)
    w.writerows(rows)

print("lab2_metrics_log.csv written:")
for r in rows:
    print(r)

lab2_metrics_log.csv written:
[1, 'index_build', '', 5676, 1, '', 265, '', '', '', '2026-05-11T13:50:28.871712', 'unique tokens=265 from 3864 filtered tokens']
[2, 'footprint_parquet', '', 5676, 1, '', '', '', '', '', '2026-05-11T13:50:28.871712', '1 parquet files']
[3, 'footprint_csv', '', 40767, 1, '', '', '', '', '', '2026-05-11T13:50:28.871712', '1 csv files, ratio parquet/csv=13.92%']
[4, 'query_lookup_hero', '', '', '', 108.16, 107, 107, '', '', '2026-05-11T13:50:28.871712', 'latency wall-clock from time.time()']
[5, 'query_lookup_ability', '', '', '', 54.18, 69, 69, '', '', '2026-05-11T13:50:28.871712', 'latency wall-clock from time.time()']
[6, 'query_lookup_damage', '', '', '', 47.55, 53, 53, '', '', '2026-05-11T13:50:28.871712', 'latency wall-clock from time.time()']
[7, 'query_lookup_mana', '', '', '', 39.76, 15, 15, '', '', '2026-05-11T13:50:28.871712', 'latency wall-clock from time.time()']
[8, 'query_lookup_ultimate', '', '', '', 49.33, 59, 59, '', '', '2026-05-11T13:50:2

## 7-bis. Verify the Parquet inverted index

In [9]:
verify_idx = spark.read.parquet("outputs/lab2/inverted_index")
print("Schema of stored inverted index:")
verify_idx.printSchema()
print(f"Total terms: {verify_idx.count()}")
verify_idx.show(10, truncate=80)

Schema of stored inverted index:
root
 |-- token: string (nullable = true)
 |-- doc_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- freq: long (nullable = true)

Total terms: 265
+----------+--------------------------------------------------------------------------------+----+
|     token|                                                                         doc_ids|freq|
+----------+--------------------------------------------------------------------------------+----+
|      hero|[hero_009, hero_010, hero_010, patch_011, patch_012, patch_013, patch_014, pa...| 107|
|      team|[match_101, match_101, match_102, match_102, match_103, match_103, match_104,...| 103|
|   ability|[hero_010, patch_011, patch_012, patch_013, patch_014, patch_015, patch_016, ...|  69|
| different|[patch_011, patch_012, patch_013, patch_014, patch_015, patch_016, patch_017,...|  61|
|adjustment|[patch_011, patch_012, patch_013, patch_014, patch_015, patch_016, patch_017,...| 

## 8. Cleanup

In [ ]:
spark.stop()
print("Done.")